### **IMPORTO LIBRERIE**

In [1]:
import pandas as pd
import numpy as np

### **IMPORTO I DATI**

In [2]:
#Importo i fogli excel come dataframe per l'analisi
df_software = pd.read_excel("datas/raw/dataset_software.xlsx", sheet_name="Risultati")
df_hardware = pd.read_excel("datas/raw/dataset_hardware.xlsx", sheet_name="Risultati")

#Stampo dimensione campione iniziale
print("Dimensione campione iniziale: " + str(len(df_software) + len(df_hardware)) + " osservazioni.")

Dimensione campione iniziale: 836 osservazioni.


### **PULISCO I DATI**

In [3]:
variabili_anni = {
        "ROE": [
            "Redditività del capitale proprio (ROE) - Netto\n2024",
            "Redditività del capitale proprio (ROE) - Netto\n2023",
            "Redditività del capitale proprio (ROE) - Netto\n2022",
            "Redditività del capitale proprio (ROE) - Netto\n2021",
            "Redditività del capitale proprio (ROE) - Netto\n2020",
        ],

        "Indice di liquidita": [
            "Indice di liquidità\n2024",
            "Indice di liquidità\n2023",
            "Indice di liquidità\n2022",
            "Indice di liquidità\n2021",
            "Indice di liquidità\n2020",
        ],

        "Debt/Equity": [
            "Debt/Equity\n2024",
            "Debt/Equity\n2023",
            "Debt/Equity\n2022",
            "Debt/Equity\n2021",
            "Debt/Equity\n2020",
        ],

        "Totale Attivo": [
            "Totale Attivo\nmigl USD 2024",
            "Totale Attivo\nmigl USD 2023",
            "Totale Attivo\nmigl USD 2022",
            "Totale Attivo\nmigl USD 2021",
            "Totale Attivo\nmigl USD 2020",
        ],
    }

Rimuovo le righe che hanno valori mancanti

In [4]:
def pulizia_valori_mancanti(df):
    df = df.copy()

    valori_da_rimuovere = ["n.d.", "n.s.", "nan", "none", ""]

    #converto le colonne in testo cosi da poter lavorare sui valori
    df_testo = (
        df.astype("string")
        .apply(lambda colonna: colonna.str.strip().str.lower())
    )

    maschera_da_tenere = pd.Series(True, index=df.index)

    #controllo che gli anni validi (per variabile) siano almeno 3
    for variabile, colonne in variabili_anni.items():
        valori_validi = ~df_testo[colonne].isin(valori_da_rimuovere)
        numero_anni_validi = valori_validi.sum(axis=1)
        maschera_da_tenere = maschera_da_tenere & (numero_anni_validi >= 3)

    df = df[maschera_da_tenere].copy()

    for colonna in df.columns:
        if df[colonna].dtype == "object":
            serie = df[colonna].astype("string").str.strip()
            df[colonna] = serie.mask(
                serie.str.lower().isin(valori_da_rimuovere),
                np.nan
            )

    return df

df_software = pulizia_valori_mancanti(df_software)
df_hardware = pulizia_valori_mancanti(df_hardware)

#Stampo la dimensione del campione statistico a disposizione dopo la pulizia
print("Dimensione campione software pulito: " + str(len(df_software)) + "osservazioni")
print("Dimensione campione hardware pulito: " + str(len(df_hardware)) + "osservazioni")
print("Dimensione campione totale pulito: " + str(len(df_software) + len(df_hardware)) + " osservazioni.")

Dimensione campione software pulito: 457osservazioni
Dimensione campione hardware pulito: 109osservazioni
Dimensione campione totale pulito: 566 osservazioni.


Converto correttamente le colonne numeriche

In [5]:
colonne_numeriche = [
    "Numero dipendenti\nUltimo anno disp.",

    "Redditività del capitale proprio (ROE) - Netto\n2024",
    "Redditività del capitale proprio (ROE) - Netto\n2023",
    "Redditività del capitale proprio (ROE) - Netto\n2022",
    "Redditività del capitale proprio (ROE) - Netto\n2021",
    "Redditività del capitale proprio (ROE) - Netto\n2020",

    "Indice di liquidità\n2024",
    "Indice di liquidità\n2023",
    "Indice di liquidità\n2022",
    "Indice di liquidità\n2021",
    "Indice di liquidità\n2020",

    "Debt/Equity\n2024",
    "Debt/Equity\n2023",
    "Debt/Equity\n2022",
    "Debt/Equity\n2021",
    "Debt/Equity\n2020",

    "Totale Attivo\nmigl USD 2024",
    "Totale Attivo\nmigl USD 2023",
    "Totale Attivo\nmigl USD 2022",
    "Totale Attivo\nmigl USD 2021",
    "Totale Attivo\nmigl USD 2020",
]

#Creo una funzione per convertire le colonne numeriche correttamente.
def converti_colonne_numeriche(df, colonne):
  df = df.copy()
  valori_mancanti = ["n.d.", "n.s.", "", "nan", "none"]

  for colonna in colonne:
    serie = df[colonna].astype("string").str.strip()
    serie = serie.mask(serie.str.lower().isin(valori_mancanti), pd.NA)

    usa_virgola_decimale = serie.str.contains(",", na=False)

    serie.loc[usa_virgola_decimale] = (
        serie.loc[usa_virgola_decimale]
        .str.replace(".", "", regex=False)
        .str.replace(",", ".", regex=False)
    )

    df[colonna] = pd.to_numeric(serie, errors="coerce")

  return df


df_hardware = converti_colonne_numeriche(df_hardware, colonne_numeriche)
df_software = converti_colonne_numeriche(df_software, colonne_numeriche)

Converto correttamente le colonne con date

In [6]:
colonne_date = [
    "Data chiusura\n2024",
    "Data chiusura\n2023",
    "Data chiusura\n2022",
    "Data chiusura\n2021",
    "Data chiusura\n2020",
    "Data di costituzione",
]

def converti_colonne_date(df, colonne):
    df = df.copy()
    for colonna in colonne:
        serie = df[colonna]

        if pd.api.types.is_datetime64_any_dtype(serie):
            df[colonna] = pd.to_datetime(serie, errors="coerce")
            continue

        date_convertite = pd.Series(pd.NaT, index=df.index, dtype="datetime64[ns]")
        valori_numerici = pd.to_numeric(serie, errors="coerce")
        maschera_seriali_excel = valori_numerici.notna()

        date_convertite.loc[maschera_seriali_excel] = pd.to_datetime(
            valori_numerici.loc[maschera_seriali_excel],
            unit="D",
            origin="1899-12-30",
            errors="coerce"
        )

        serie_testo = serie.astype("string").str.strip()
        maschera_testo = ~maschera_seriali_excel & serie_testo.notna()

        date_convertite.loc[maschera_testo] = pd.to_datetime(
            serie_testo.loc[maschera_testo],
            format="%Y-%m-%d %H:%M:%S",
            errors="coerce"
        )

        maschera_solo_data = maschera_testo & date_convertite.isna()
        date_convertite.loc[maschera_solo_data] = pd.to_datetime(
            serie_testo.loc[maschera_solo_data],
            format="%Y-%m-%d",
            errors="coerce"
        )

        df[colonna] = date_convertite

    return df

df_hardware = converti_colonne_date(df_hardware, colonne_date)
df_software = converti_colonne_date(df_software, colonne_date)

Trasformo i "Total assets" in valori logaritmici per normalizzare le distribuzione

In [7]:
colonne_attivo = [
    "Totale Attivo\nmigl USD 2024",
    "Totale Attivo\nmigl USD 2023",
    "Totale Attivo\nmigl USD 2022",
    "Totale Attivo\nmigl USD 2021",
    "Totale Attivo\nmigl USD 2020",
]


def log_attivo(df, colonne_attivo):
    df = df.copy()

    attivo = df[colonne_attivo].astype(float)

    # Tengo solo valori strettamente positivi
    attivo_positivo = attivo.where(attivo > 0)

    # Calcolo il log naturale solo sui valori positivi
    log_attivo = np.log(attivo_positivo)

    # Calcolo la media dei log per ogni azienda
    df["log_attivo_medio"] = log_attivo.mean(axis=1, skipna=True)

    return df


df_hardware = log_attivo(df_hardware, colonne_attivo)
df_software = log_attivo(df_software, colonne_attivo)

# Salvo una copia prima della rimozione outlier per poter confrontare diversi criteri
df_hardware_pre_outlier = df_hardware.copy()
df_software_pre_outlier = df_software.copy()

Pulizia outlier (1): IRQ sulle medie delle variabili

In [8]:
def rimuovi_outlier_iqr_su_medie(
    df,
    colonne_roe,
    colonne_debt_equity,
    colonne_liquidita):

    df = df.copy()

    # Calcolo delle medie per azienda
    df["roe_medio"] = df[colonne_roe].mean(axis=1, skipna=True)
    df["debt_equity_medio"] = df[colonne_debt_equity].mean(axis=1, skipna=True)
    df["liquidita_media"] = df[colonne_liquidita].mean(axis=1, skipna=True)

    colonne_medie = [
        "roe_medio",
        "debt_equity_medio",
        "liquidita_media"
    ]

    maschera_da_tenere = pd.Series(True, index=df.index)

    for colonna in colonne_medie:
        q1 = df[colonna].quantile(0.25)
        q3 = df[colonna].quantile(0.75)
        iqr = q3 - q1

        limite_inferiore = q1 - 1.5 * iqr
        limite_superiore = q3 + 1.5 * iqr

        maschera_colonna = (
            (df[colonna] >= limite_inferiore) &
            (df[colonna] <= limite_superiore)
        )

        maschera_da_tenere = maschera_da_tenere & maschera_colonna

    df = df[maschera_da_tenere].copy()

    return df

df_hardware = rimuovi_outlier_iqr_su_medie(df_hardware, variabili_anni["ROE"], variabili_anni["Debt/Equity"], variabili_anni["Indice di liquidita"])
df_software = rimuovi_outlier_iqr_su_medie(df_software, variabili_anni["ROE"], variabili_anni["Debt/Equity"], variabili_anni["Indice di liquidita"])

print("Dimensione campione software pulito: " + str(len(df_software)) + " osservazioni")
print("Dimensione campione hardware pulito: " + str(len(df_hardware)) + " osservazioni")
print("Dimensione campione totale pulito: " + str(len(df_software) + len(df_hardware)) + " osservazioni.")

Dimensione campione software pulito: 336 osservazioni
Dimensione campione hardware pulito: 79 osservazioni
Dimensione campione totale pulito: 415 osservazioni.


Pulizia outlier (2): winsorizzazione al 5 e 95esimo percentile sulle medie delle variabili

In [9]:
def winsorizza_outlier_su_medie(
    df,
    colonne_roe,
    colonne_debt_equity,
    colonne_liquidita):

    df = df.copy()

    # Calcolo delle medie per azienda
    df["roe_medio"] = df[colonne_roe].mean(axis=1, skipna=True)
    df["debt_equity_medio"] = df[colonne_debt_equity].mean(axis=1, skipna=True)
    df["liquidita_media"] = df[colonne_liquidita].mean(axis=1, skipna=True)

    colonne_medie = [
        "roe_medio",
        "debt_equity_medio",
        "liquidita_media"
    ]

    for colonna in colonne_medie:
        limite_inferiore = df[colonna].quantile(0.05)
        limite_superiore = df[colonna].quantile(0.95)

        df[colonna] = df[colonna].clip(
            lower=limite_inferiore,
            upper=limite_superiore
        )

    return df

df_hardware_outlier_medi_winsorizzati = winsorizza_outlier_su_medie(df_hardware_pre_outlier, variabili_anni["ROE"], variabili_anni["Debt/Equity"], variabili_anni["Indice di liquidita"])
df_software_outlier_medi_winsorizzati = winsorizza_outlier_su_medie(df_software_pre_outlier, variabili_anni["ROE"], variabili_anni["Debt/Equity"], variabili_anni["Indice di liquidita"])

print("Dimensione campione software con winsorizzazione sulle medie: " + str(len(df_software_outlier_medi_winsorizzati)) + " osservazioni")
print("Dimensione campione hardware con winsorizzazione sulle medie: " + str(len(df_hardware_outlier_medi_winsorizzati)) + " osservazioni")
print("Dimensione campione totale con winsorizzazione sulle medie: " + str(len(df_software_outlier_medi_winsorizzati) + len(df_hardware_outlier_medi_winsorizzati)) + " osservazioni.")

Dimensione campione software con winsorizzazione sulle medie: 457 osservazioni
Dimensione campione hardware con winsorizzazione sulle medie: 109 osservazioni
Dimensione campione totale con winsorizzazione sulle medie: 566 osservazioni.


Pulizia outliers (3): IQR sui singoli valori annuali

In [10]:
def rimuovi_outlier_iqr_su_singoli_valori(
    df,
    colonne_roe,
    colonne_debt_equity,
    colonne_liquidita):

    df = df.copy()

    colonne_outlier = (
        colonne_roe +
        colonne_debt_equity +
        colonne_liquidita
    )

    maschera_da_tenere = pd.Series(True, index=df.index)

    for colonna in colonne_outlier:
        valori_validi = df[colonna].dropna()

        q1 = valori_validi.quantile(0.25)
        q3 = valori_validi.quantile(0.75)
        iqr = q3 - q1

        limite_inferiore = q1 - 1.5 * iqr
        limite_superiore = q3 + 1.5 * iqr

        maschera_colonna = (
            df[colonna].isna() |
            (
                (df[colonna] >= limite_inferiore) &
                (df[colonna] <= limite_superiore)
            )
        )

        maschera_da_tenere = maschera_da_tenere & maschera_colonna

    df = df[maschera_da_tenere].copy()

    df["roe_medio"] = df[colonne_roe].mean(axis=1, skipna=True)
    df["debt_equity_medio"] = df[colonne_debt_equity].mean(axis=1, skipna=True)
    df["liquidita_media"] = df[colonne_liquidita].mean(axis=1, skipna=True)

    return df


df_hardware_outlier_singoli = rimuovi_outlier_iqr_su_singoli_valori(df_hardware_pre_outlier, variabili_anni["ROE"], variabili_anni["Debt/Equity"], variabili_anni["Indice di liquidita"])
df_software_outlier_singoli = rimuovi_outlier_iqr_su_singoli_valori(df_software_pre_outlier, variabili_anni["ROE"], variabili_anni["Debt/Equity"], variabili_anni["Indice di liquidita"])

print("Dimensione campione software con outlier sui singoli valori: " + str(len(df_software_outlier_singoli)) + " osservazioni")
print("Dimensione campione hardware con outlier sui singoli valori: " + str(len(df_hardware_outlier_singoli)) + " osservazioni")
print("Dimensione campione totale con outlier sui singoli valori: " + str(len(df_software_outlier_singoli) + len(df_hardware_outlier_singoli)) + " osservazioni.")

Dimensione campione software con outlier sui singoli valori: 221 osservazioni
Dimensione campione hardware con outlier sui singoli valori: 58 osservazioni
Dimensione campione totale con outlier sui singoli valori: 279 osservazioni.


### **SALVO I DATASET PULITI**

In [11]:
df_hardware.to_csv("datas/processed/dataset_hardware_outlier_medi.csv", index=False)
df_software.to_csv("datas/processed/dataset_software_outlier_medi.csv", index=False)

df_hardware_outlier_medi_winsorizzati.to_csv("datas/processed/dataset_hardware_outlier_medi_winsorizzati.csv", index=False)
df_software_outlier_medi_winsorizzati.to_csv("datas/processed/dataset_software_outlier_medi_winsorizzati.csv", index=False)

df_hardware_outlier_singoli.to_csv("datas/processed/dataset_hardware_outlier_singoli.csv", index=False)
df_software_outlier_singoli.to_csv("datas/processed/dataset_software_outlier_singoli.csv", index=False)